In [ ]:
# 每日把主表超過 retention 的高頻資料移到 archive
from database import get_engine
from datetime import datetime, timedelta, timezone
from config_intervals import MAIN_TABLE_RETENTION_DAYS
from sqlalchemy import text


def archive_old_intraday(batch_limit: int = None):
    """
    把主表超過 retention 的高頻資料移到 archive。
    若 batch_limit 提供（int），會分批搬移（每次 INSERT/DELETE 限制筆數），避免長 transaction。
    """
    engine = get_engine()
    now = datetime.now(timezone.utc)
    with engine.begin() as conn:
        for interval, keep_days in MAIN_TABLE_RETENTION_DAYS.items():
            if keep_days >= 99999:
                continue
            cutoff = now - timedelta(days=keep_days)

            insert_sql = f"""
            INSERT INTO stock_intraday_archive (symbol, data_type, ts, open, high, low, close, volume)
            SELECT symbol, data_type, ts, open, high, low, close, volume
            FROM stock_data
            WHERE data_type = :interval AND ts < :cutoff
            ON CONFLICT ON CONSTRAINT uq_archive_symbol_datatype_ts DO NOTHING
            """
            conn.execute(text(insert_sql), {"interval": interval, "cutoff": cutoff})

            delete_sql = """
            DELETE FROM stock_data
            WHERE data_type = :interval AND ts < :cutoff
            """
            conn.execute(text(delete_sql), {"interval": interval, "cutoff": cutoff})

            print(f"Archived interval={interval} before {cutoff.isoformat()}")